# picoNewton_v4 — Production Model Solution
This is the single Colab entry point for the six-artery anisotropic Womersley → membrane → Piezo1 calculation. Run all cells once from top to bottom. It creates a timestamped Google Drive run, installs the pinned package, solves the publication-resolution model, executes the hypothesis/control matrix, generates figures and tables, and archives provenance and checksums.

**No quick mode, dry run, reduced-resolution simulation, or pytest stage is executed.**


## Scientific question
Does anisotropic near-wall forcing provide mechanosensory information distinct from wall shear stress across Aortic Root, Thoracic Aorta, Femoral, Carotid, Iliac, and Brachial arteries?

The workflow keeps wall shear stress, signed transverse Lamb force, and nonnegative Lamb-force exposure physically separate. Signed force is directional; exposure is never treated as a signed traction.


## Vector-resolved membrane and Piezo1 model
Normal and tangential mechanics are transferred through separate passive viscoelastic branches to apical and junctional membrane tension. The four-state Piezo1 model reports open probability, current, and a reduced-order calcium-scale endpoint. Predictions remain conditional on the packaged inputs, constitutive assumptions, transfer parameters, and calibration.


## Fixed production resolution
The calculation is fixed at radial order 150, 2,048 time points per cardiac cycle, 256 near-wall integration points, six arteries, the complete control matrix, and the configured sensitivity analysis. The package API names this resolution `full`; it is not a user-selectable mode here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL='https://github.com/khalid-saqr/picoNewton.git'
REPO_REF='8fd376075e90a739ca74f1afede8a895344b3054'
PACKAGE_SUBDIR='picoNewton_v4'
CALIBRATION_RELATIVE_PATH='configs/literature_calibration.json'
NUMERICAL_PROFILE='full'
RUN_SENSITIVITY_SCAN=True
MINIMUM_PASSING_ARTERIES=4
EXPECTED_ARTERIES={'Aortic Root','Thoracic Aorta','Femoral','Carotid','Iliac','Brachial'}


## Timestamped runtime
Each execution writes to `MyDrive/picoNewton_v4_runtime/runs/YYYYMMDD_HHMMSS_UTC_<suffix>/`. Colab local storage is used for computation, then the complete result is copied to Drive. Existing runs are never overwritten.


In [ ]:
from __future__ import annotations
import hashlib,json,os,platform,secrets,shutil,subprocess,sys
from datetime import datetime,timezone
from pathlib import Path
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='2'
RUN_ID=f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')}_{secrets.token_hex(2)}"
DRIVE_ROOT=Path('/content/drive/MyDrive/picoNewton_v4_runtime')
RUN_DIR=DRIVE_ROOT/'runs'/RUN_ID
LOCAL_ROOT=Path('/content/picoNewton_v4_work')/RUN_ID
REPO_DIR=LOCAL_ROOT/'repository'
PACKAGE_DIR=REPO_DIR/PACKAGE_SUBDIR
LOCAL_OUTPUT=LOCAL_ROOT/'model_outputs'
for p in (RUN_DIR/'inputs',RUN_DIR/'configs',RUN_DIR/'logs',RUN_DIR/'provenance',RUN_DIR/'environment'): p.mkdir(parents=True,exist_ok=True)
print('Run ID:',RUN_ID); print('Drive output:',RUN_DIR)


## Install the pinned package
The exact Git commit is cloned and installed. Because the project uses a `src/` package layout and Colab keeps the same Python kernel alive after `pip`, the installation cell explicitly adds `<package>/src` to the current kernel and verifies the import before continuing. Installation or import failure stops execution before the model starts. No package test suite is run.


In [ ]:
def run_logged(cmd,log,cwd=None):
    cmd=[str(x) for x in cmd]
    r=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    _=Path(log).write_text('$ '+' '.join(cmd)+'\n\n'+r.stdout,encoding='utf-8')
    print(r.stdout[-3000:])
    if r.returncode: raise RuntimeError(f'Command failed ({r.returncode}): {cmd}')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_logged(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,REPO_DIR],RUN_DIR/'logs'/'git_clone.log')
run_logged(['git','-C',REPO_DIR,'fetch','--depth','1','origin',REPO_REF],RUN_DIR/'logs'/'git_fetch.log')
run_logged(['git','-C',REPO_DIR,'checkout','--detach','FETCH_HEAD'],RUN_DIR/'logs'/'git_checkout.log')

GIT_SHA=subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()
if GIT_SHA!=REPO_REF:
    raise RuntimeError(f'Commit mismatch: {GIT_SHA}')
if not PACKAGE_DIR.is_dir():
    raise FileNotFoundError(PACKAGE_DIR)

run_logged(
    [sys.executable,'-m','pip','install','--no-deps','--force-reinstall',PACKAGE_DIR],
    RUN_DIR/'logs'/'pip_install.log',
)

# The repository uses the standard src-layout. Expose it to this already-running
# Colab kernel immediately; adding PACKAGE_DIR itself is not sufficient.
SOURCE_DIR=PACKAGE_DIR/'src'
if not (SOURCE_DIR/'piconewton_v4'/'__init__.py').is_file():
    raise FileNotFoundError(SOURCE_DIR/'piconewton_v4'/'__init__.py')
source_text=str(SOURCE_DIR.resolve())
if source_text not in sys.path:
    sys.path.insert(0,source_text)

import importlib
importlib.invalidate_caches()
piconewton_v4=importlib.import_module('piconewton_v4')
if piconewton_v4.__version__!='4.0.0':
    raise RuntimeError(f'Unexpected package version: {piconewton_v4.__version__}')

_=(RUN_DIR/'provenance'/'git_commit.txt').write_text(GIT_SHA+'\n',encoding='utf-8')
print('Imported piconewton_v4:',piconewton_v4.__version__)
print('Import path:',Path(piconewton_v4.__file__).resolve())


## Freeze scientific inputs
Before solving, the notebook confirms that the model configuration, calibration, and artery table exist and that the table contains exactly the required six arteries. These are input-integrity checks, not a preliminary simulation.


In [ ]:
import pandas as pd
import piconewton_v4
from piconewton_v4.calibration import load_parameterization
CALIBRATION_PATH=PACKAGE_DIR/CALIBRATION_RELATIVE_PATH
MODEL_CONFIG_PATH=PACKAGE_DIR/'configs'/'model.json'
ARTERY_INPUT_PATH=PACKAGE_DIR/'data'/'ground_truth_arteries.csv'
for p in (CALIBRATION_PATH,MODEL_CONFIG_PATH,ARTERY_INPUT_PATH):
    if not p.is_file(): raise FileNotFoundError(p)
arteries=pd.read_csv(ARTERY_INPUT_PATH)
col=next((c for c in arteries.columns if c.strip().lower() in {'artery','artery_name','name'}),None)
if col is None or set(arteries[col].astype(str).str.strip())!=EXPECTED_ARTERIES: raise ValueError('Six-artery input contract failed')
model_configuration=json.loads(MODEL_CONFIG_PATH.read_text())
interface_parameters,endpoint_parameters,calibration_audit=load_parameterization(CALIBRATION_PATH)
for src,dst in ((ARTERY_INPUT_PATH,RUN_DIR/'inputs'/ARTERY_INPUT_PATH.name),(MODEL_CONFIG_PATH,RUN_DIR/'configs'/MODEL_CONFIG_PATH.name),(CALIBRATION_PATH,RUN_DIR/'configs'/CALIBRATION_PATH.name)): shutil.copy2(src,dst)
input_record={'run_id':RUN_ID,'git_sha':GIT_SHA,'package_version':piconewton_v4.__version__,'profile':NUMERICAL_PROFILE,'sensitivity_scan':RUN_SENSITIVITY_SCAN,'arteries':sorted(EXPECTED_ARTERIES),'calibration_status':calibration_audit.get('calibration_status'),'missing_source_groups':calibration_audit.get('missing_source_groups',[])}
(RUN_DIR/'provenance'/'input_record.json').write_text(json.dumps(input_record,indent=2,default=str)+'\n')
print(json.dumps(input_record,indent=2))


## Solve the production model
This is the only numerical solution cell. It runs all six arteries at the fixed publication resolution and executes the configured sensitivity analysis.


In [ ]:
from piconewton_v4.workflow import run_workflow
manifest=run_workflow(package_root=PACKAGE_DIR,output_root=LOCAL_OUTPUT,profile=NUMERICAL_PROFILE,calibration_path=CALIBRATION_PATH,run_scan=RUN_SENSITIVITY_SCAN)
if manifest.get('status') not in {'complete','completed','passed'}: raise RuntimeError(manifest)
print('Workflow status:',manifest.get('status'))


## Validate and classify the completed result
The completed outputs are checked for six-artery completeness, probability conservation, passive mechanics, and correct separation of signed force from exposure. Detection thresholds are then applied and the standard figures are generated.


In [ ]:
from piconewton_v4.hypotheses import DecisionThresholds,write_decisions
from piconewton_v4.reporting import generate_standard_figures
from piconewton_v4.validation import validate_output_directory
validation_report=validate_output_directory(LOCAL_OUTPUT)
if not validation_report.get('passed'): raise RuntimeError(validation_report)
thresholds=DecisionThresholds(current_rms_pa=endpoint_parameters.current_detection_limit_pa,calcium_rms_nm=endpoint_parameters.calcium_detection_limit_nm,minimum_arteries=MINIMUM_PASSING_ARTERIES)
decisions=write_decisions(LOCAL_OUTPUT/'hypothesis_effects.csv',LOCAL_OUTPUT/'hypothesis_decisions.csv',thresholds,LOCAL_OUTPUT/'decision_thresholds.json')
figure_paths=generate_standard_figures(LOCAL_OUTPUT,LOCAL_OUTPUT/'figures')
(LOCAL_OUTPUT/'output_validation.json').write_text(json.dumps(validation_report,indent=2)+'\n')
print(json.dumps(validation_report,indent=2)); print('Figures:',len(figure_paths))


## Principal tables
The artery summary, effect sizes, and hypothesis decisions are displayed below. Complete arrays and all generated files are archived.


In [ ]:
from IPython.display import display
for name in ('summary.csv','hypothesis_effects.csv','hypothesis_decisions.csv'):
    p=LOCAL_OUTPUT/name
    if not p.is_file(): raise FileNotFoundError(p)
    print(name); display(pd.read_csv(p))


## Archive and checksum
The complete native output tree, Python environment, run manifest, and SHA-256 checksums are stored in the timestamped Google Drive directory.


In [ ]:
(RUN_DIR/'environment'/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
system={'run_id':RUN_ID,'completed_utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'platform':platform.platform(),'package_version':piconewton_v4.__version__,'git_sha':GIT_SHA,'profile':NUMERICAL_PROFILE,'sensitivity_scan':RUN_SENSITIVITY_SCAN,'workflow_status':manifest.get('status'),'validation_passed':validation_report.get('passed'),'figure_count':len(figure_paths)}
(RUN_DIR/'environment'/'system.json').write_text(json.dumps(system,indent=2,default=str)+'\n')
dst=RUN_DIR/'model_outputs'
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(LOCAL_OUTPUT,dst)
(RUN_DIR/'provenance'/'run_manifest.json').write_text(json.dumps({**system,'arteries':sorted(EXPECTED_ARTERIES),'calibration_file':CALIBRATION_RELATIVE_PATH},indent=2,default=str)+'\n')
def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
checks={str(p.relative_to(RUN_DIR)):sha256(p) for p in sorted(RUN_DIR.rglob('*')) if p.is_file()}
(RUN_DIR/'provenance'/'SHA256SUMS.json').write_text(json.dumps(checks,indent=2,sort_keys=True)+'\n')
print('Production run complete:',RUN_ID); print('Persistent output:',RUN_DIR); print('Files checksummed:',len(checks))
